In [38]:
#=
    Various functions for creation of operators in ItensorMPS.jl. 
    - Extended Hubbard model Hamiltonian.
    - Density operators 
    - Random ansatz for initial states in phases such as 
        - Metallic 
        - Charge density wave
        - Spin density wave
=#
using ITensors
using ITensorMPS
using Random
#=
    Generates the MPO for the EHM Hamiltonian 
    with strengths J, U and V. 
    Requires a SiteType sites.
=#
function H_EHM(N, J, U, V, sites)
    os = OpSum()
    for i in 1:(N - 1)
      # Knetic 
      os -= J, "Cdagup", i, "Cup", i + 1
      os -= J, "Cdagup", i + 1, "Cup", i
      os -= J, "Cdagdn", i, "Cdn", i + 1
      os -= J, "Cdagdn", i + 1, "Cdn", i
      # Nearest-neighbours
      os += V, "Ntot", i, "Ntot", i + 1
    end
    # on-site
    for i in 1:N
      os += U, "Nupdn", i
    end
    return MPO(os, sites)
end


#= 
    Builds the 1-particle reduced density matrix. 
    
    This matrix depends on just two correlators so it is 
    easily implemented by the correlation_matrix from ITensorMPS. 

=#
function build_1_particle_rdm(state) 
    L = length(state)
    #=  
        N and not L since any site can have spin up or down
    =#
    rho_1 = zeros(ComplexF64, 2*L, 2*L) 

    Cupup = correlation_matrix(state, "Cdagup", "Cup")
    Cdndn = correlation_matrix(state, "Cdagdn", "Cdn")

    Cupdn = correlation_matrix(state, "Cdagup", "Cdn")
    # Cdnup = correlation_matrix(state, "Cdagdn", "Cup")   

    for i in 1:L
        for j in 1:L
            rho_1[i, j] = Cupup[i,j]
            rho_1[i, j + L] = Cupdn[i,j]
            rho_1[i + L, j] = Cupdn[i, j]' # Cdnup[i,j]
            rho_1[i + L, j + L] = Cdndn[i,j]
        end 
    end
    return rho_1 = rho_1 / L
end
#=
    Create the basis for pairs of spins 
    1 = (1, up) , 2 = (1, dn), 3 = (2, up), ...
=#
function two_fermion_basis_pairs(L)
    num_modes = 2 * L
    pairs = []
    for m1 in 1:num_modes
        for m2 in (m1 + 1):num_modes
            push!(pairs, (m1, m2))
        end
    end
    return pairs
end

#=
    Gets the indexed site and spin.
=#
function mode_to_site_spin(m::Int)
    site = (m + 1) ÷ 2
    spin = isodd(m) ? "up" : "dn"
    return site, spin
end
#= 
    Returns a two-particle reduced density matrix. 

    This function computes only the part of the matrix in which 
    all of the correlators are different. 
    
    Tolerance of the julia language packages are very low, so in general
    this computation gives various non-hermitian matrices. 
=#
function build_2_particle_rdm(phi, sites)

    L = length(phi)
    
    # Basis for pairs of fermions.
    pairs = two_fermion_basis_pairs(L)

    dim = length(pairs)

    # @show dim

    rho_2 = zeros(ComplexF64, dim, dim)
    #= 
        Loops through all the configurations with no repeated indices and spins, 
        stores the elements rho_2[p, q] = <psi' | O | psi>. 
    =#
    for (p, (i, j)) in enumerate(pairs)
        # @show (p, (i, j))

        i_site, i_spin = mode_to_site_spin(i)
        j_site, j_spin = mode_to_site_spin(j)

        # @show (mode_to_site_spin(i), mode_to_site_spin(j))

        for (q, (k, l)) in enumerate(pairs)
            # @show (q, (k, l))
           
            os = OpSum() 

            k_site, k_spin = mode_to_site_spin(k)
            l_site, l_spin = mode_to_site_spin(l)

            # @show (mode_to_site_spin(k), mode_to_site_spin(l))

            os += "Cdag$i_spin", i_site, "Cdag$j_spin", j_site, "C$l_spin", l_site, "C$k_spin", k_site

            O_mpo = MPO(os, sites)

            rho_2[p, q] = inner(phi', O_mpo, phi)
        end
    end
    # Force hermiticity
    rho_2 = (rho_2 + rho_2') / 2.0

    # Normalize by remaining number of particles 
    rho_2 = (2.0 / (L*(L-1))) * rho_2
    return rho_2
end
#=
    Returns the density of up and down 
    electrons.
=#
function density_operators(N, psi, sites)
    upd = fill(0.0, N)
    dnd = fill(0.0, N)
    updn = fill(0.0, N) 
    for j in 1:N
        orthogonalize!(psi, j)
        psidag_j = dag(prime(psi[j], "Site"))
        upd[j] = scalar(psidag_j * op(sites, "Nup", j) * psi[j])
        dnd[j] = scalar(psidag_j * op(sites, "Ndn", j) * psi[j])
        updn[j] = scalar(psidag_j * op(sites, "Nupdn", j) * psi[j])
    end
    return upd, dnd, updn
end
using LinearAlgebra
# Von Neumann entropy in bits = configuration space.
function von_neumann_entropy(vals; atol=1e-12)
    return - sum(vals .* log.(vals)), - sum(vals .* log2.(vals))
end

function quantum_coherence(rho)
    dim = size(rho)[1]
    C = 0.0
    for i in 1:dim
        for j in (i+1):dim
            # maybe return here and force hermitian()
            C += abs(rho[i, j])
        end
    end
    return C 
end
#=
    Diagonalizes the matrix rho as 
    rho = \sum_i e^{-\xi_i} with x_{i+1} >= x_i 
    and returns the difference from the 
=#
function entanglement_gap(eigenvals; cutoff=1e-12)

    sorted_indices = sortperm(eigvals, rev=true)
    eigvals_sorted = eigvals[sorted_indices]

    eigvals_clipped = max.(eigvals_sorted, cutoff)
    entanglement_spectrum = -log.(eigvals_clipped)
    gaps = diff(entanglement_spectrum)

    return gaps, entanglement_spectrum
end

function entanglement_gap(eigvals; cutoff=1e-12)

    sorted_indices = sortperm(eigvals, rev=true)
    eigvals_sorted = eigvals[sorted_indices]
    eigvals_clipped = max.(eigvals_sorted, cutoff)

    xis = -log.(eigvals_clipped)

    return xis
end


using Statistics
function average_single_site_entanglement(L, up, dn, updn)
    single_site_entanglement = fill(0.0, L)
    for i in 1:L 
        w_2 = updn[i] 
        w_up = up[i] - w_2 
        w_dn = dn[i] - w_2 
        w_0 = 1 - w_up - w_dn - w_2 
        single_site_entanglement[i] = 1 - (w_2^2 + w_up^2 + w_dn^2 + w_0^2)
    end
    return Statistics.mean(single_site_entanglement)
end 

function m_sdw(L, Sj)
    m_sdw_val = 0.0 
    for j in 1:L 
        m_sdw_val += (-1)^(j-1) * Sj[j]
    end
    return m_sdw_val / L
end
function m_cdw(L, nj)
    m_cdw_val = 0 
    for j in 1:L 
        m_cdw_val += (-1)^(j-1) * (nj[j] - 1)
    end
    return m_cdw_val / L 
end 

function compute_GS_measures(L, sites, psi; cutoff=1e-12)
    #=
        Can reconstruct the densities from these three 
        results.
    =#
    upd, dnd, updn = density_operators(L, psi, sites)

    @show upd
    @show dnd
    @show updn

    charge_density = upd .+ dnd
    # removed the 1/2 factor
    magnetization = (upd .- dnd) / 2
    
    # compute order parameters
    op_m_cdw = abs(m_cdw(L, charge_density))
    op_m_sdw = abs(m_sdw(L, magnetization))               

    #= 
        Implementation of average single-site entanglement 
            \mathcal{L} = 1 - (1/L) \sum_{i} Tr (rho_i^2)
    =#
    single_site_entanglement = average_single_site_entanglement(L, upd, dnd, updn)

    # Reduced density matrix computations
    # one-particle RDM
    
    rho_1 = build_1_particle_rdm(psi)

    # diagonalize rho_1 
    vals_1 = eigen(Hermitian(rho_1)).values

    # Compute entanglement spectrum 
    sorted_indices = sortperm(vals_1, rev=true)
    eigvals_sorted = vals_1[sorted_indices]
    eigvals_clipped = max.(eigvals_sorted, cutoff)
    Omega_1rdm = -log.(eigvals_clipped)

    # Shifted Von Neumann entropy 
    pos_vals = vals_1[vals_1 .> cutoff]
    S_1rdm, S_1rdm_bits = - sum(vals_1 .* log.(vals_1)), - sum(vals_1 .* log2.(vals_1))
   
    E_p, E_p_bits = S_1rdm - log(L), S_1rdm_bits - log2(L)
    
    # quantum coherence
    coh_1rdm = sum(abs, rho_1) - sum(abs, diag(rho_1))

    # two-particle RDM
    rho_2 = build_2_particle_rdm(psi, sites)

    # diagonalize rho_2
    vals_2 = eigen(Hermitian(rho_2)).values

    # Compute entanglement spectrum 
    sorted_indices = sortperm(vals_2, rev=true)
    eigvals_sorted = vals_2[sorted_indices]
    eigvals_clipped = max.(eigvals_sorted, cutoff)
    Omega_2rdm = -log.(eigvals_clipped)
    
    # Shifted Von Neumann
    pos_vals = vals_2[vals_2 .> cutoff]
    S_2rdm, S_2rdm_bits = - sum(vals_2 .* log.(vals_2)), - sum(vals_2 .* log2.(vals_2))

    Q_2, Q_2_bits  = S_2rdm - log(L*(L-1)/2), S_2rdm_bits - log2(L*(L-1)/2)

    # quantum coherence
    coh_2rdm = sum(abs, rho_2) - sum(abs, diag(rho_2))

    dict = Dict(
        "charge_density" => charge_density, 
        "magnetization" => magnetization,
        "doublons" => updn,
        "E_p" => E_p, 
        "E_p_bits" => E_p_bits,
        "single_site_entanglement" => single_site_entanglement,
        "coh_1rdm" => coh_1rdm,
        "Omega_1rdm" => Omega_1rdm,
        "coh_2rdm" => coh_2rdm,
        "Q_2" => Q_2, 
        "Q_2_bits" => Q_2_bits,
        "Omega_2rdm" => Omega_2rdm,
        "op_m_sdw" => op_m_sdw,
        "op_m_cdw" => op_m_cdw, 
    ) 
    return dict
end
#= 
    A collection of functions to generate 
    to generate random initial product states  
    using the random_mps method. 
    
    These states corresponds to diferent 
    phases of matter and are used according 
    to the phase diagram of the EHM.
=#

function random_metallic_state(L, Nup, Ndn)
    state = fill("Emp", L)
    p = Nup + Ndn
    for i in 1:L
        j = L - i
        if(p > j)
            state[j] = "UpDn"
            p -= 2
        elseif (p > 0) 
            state[j] = j % 2 == 1 ? "Up" : "Dn"
            p -= 1
        end
    end
    return state
end

function random_cdw_state(L, Nup, Ndn)
    state = fill("Emp", L)
    Nup_extra = Int.(Nup % Ndn)
    for i in 1:2:(L - Nup_extra)
        state[i] = "UpDn"
        Nup-=1
    end
    if Nup != 0
        state[L] = "Up"
    end
    return state
end 
function random_sdw_state(L, Nup, Ndn)
    state = fill("Emp", L)
    Nup_extra = Int.(Nup % Ndn)
    for i=1:2:(L-Nup_extra)
        state[i] = "Up"
        state[i+1] = "Dn"
    end
    if Nup != 0 
        state[L] = "Up"
    end
    return state 
end


function random_ps_state(L, Nup, Ndn)
    state = fill("Emp", L)
    
    Ndbl = min(Nup, Ndn)
    Nup -= Ndbl
    Ndn -= Ndbl
    
    for i in 1:Ndbl
        state[i] = "UpDn"
    end
    
    next_site = Ndbl + 1
    
    if Nup > 0
        state[next_site] = "Up"
        next_site += 1
    elseif Ndn > 0
        state[next_site] = "Dn"
        next_site += 1
    end
    return state
end

function product_ps_state(L::Int, Nup::Int, Ndn::Int; cluster_side::Symbol=:left)
    state = fill("Emp", L)
    Ndbl = min(Nup, Ndn)
    Nup_rem = Nup - Ndbl
    Ndn_rem = Ndn - Ndbl

    # build a vector of occupation labels for the particles clustered
    labels = String[]
    # first put doublons
    for i in 1:Ndbl
        push!(labels, "UpDn")
    end
    # then all remaining Ups
    for i in 1:Nup_rem
        push!(labels, "Up")
    end
    # then all remaining Dns
    for i in 1:Ndn_rem
        push!(labels, "Dn")
    end

    # put cluster at left or right (or center if you want)
    if cluster_side == :left
        for (i, lab) in enumerate(labels)
            state[i] = lab
        end
    elseif cluster_side == :right
        for (i, lab) in enumerate(labels)
            state[L - length(labels) + i] = lab
        end
    else
        # put cluster centered (simple version)
        start = Int(floor((L - length(labels))/2)) + 1
        for (i, lab) in enumerate(labels)
            state[start + i - 1] = lab
        end
    end

    return state
end


#= 
    Returns a state for the point (U, V) of the Extended Hubbard Model.
=#
function state_ehm_diagram(L, Nup, Ndn, U, V)

    state = fill("Emp", L)
    
    region = get_region(U, V)

    # Weak coupling = metallic
    if region == "METALLIC" 
        state = random_metallic_state(L, Nup, Ndn)
    # CDW
    elseif region == "CDW"
        state = random_cdw_state(L, Nup, Ndn)
    elseif region == "SDW"
        state = random_sdw_state(L, Nup, Ndn)
    else
        state = product_ps_state(L, Nup, Ndn) # random_ps_state(L, Nup, Ndn)
        @show state
    end
    return state
end
#= 
    Trying to separate the phase-diagram states not only 
    in big squared blocks.
    See notebook ploting_ehm_diagram_selection_of_states.ipynb for an example of plot 
    using this function. I try to made it look like the plot schematic 
    diagram for the EHM. 
=#
function get_region(u, v)

    alpha = 0.5

    # Boundary functions
    get_metallic_lower_boundary(u) = u <= 0 ? -exp(alpha * u) : -alpha * u - 1.0
    get_metallic_upper_boundary_h(u) = -0.1 * u
    get_metallic_upper_boundary_negative(u) = -2.5 * alpha * u

    lower = get_metallic_lower_boundary(u)
    upper_h = get_metallic_upper_boundary_h(u)
    upper_neg = get_metallic_upper_boundary_negative(u)

    if v <= lower
        return "PS"
    elseif (u < 0 && v < 0 && v > lower) ||
           (u > 0 && v > 0 && v < upper_h) ||
           (u > 0 && v < 0 && v > lower && v < upper_neg)
        return "METALLIC"
    elseif (u < 0 && v > 0) ||
           (u > 0 && v > 0 && v >= 2 * u)
        return "CDW"
    elseif (u > 0 && v < 0 && v > lower) ||
           (u > 0 && v > 0 && v >= upper_h && v < 2 * u)
        return "SDW"
    end
    return "PS"
end

get_region (generic function with 1 method)

In [42]:
L = 16

Npart = floor(Int, L/2)
Nup = Npart + L % 2
Ndn = L - Nup

J = 1.0
U = -6.0
V = -4.0

sites = siteinds("Electron", L; conserve_qns=true)

H = H_EHM(L, J, U, V, sites)
#=
The best state for variational step of the DMRG algorithm.
=#
state = state_ehm_diagram(L, Nup, Ndn, U, V)
# dmrg parameters 
nsweeps = 10
m = 16
maxdim = [50, 100, 200, 400, 800, 800, 1200, 1400]
cutoff = [1e-6, 1e-8, 1e-9, 1e-10, 1e-12, 1e-12, 1e-14, 1e-14]
psi0 = random_mps(sites, state; linkdims=m)
# Start DMRG calculation:
energy, psi = dmrg(H, psi0; nsweeps, maxdim=maxdim, cutoff=cutoff)

dict_results = compute_GS_measures(L, sites, psi0)
dict_results["energy"]  = energy
@show dict_results["E_p"]
@show dict_results["Q_2"]

state = ["UpDn", "UpDn", "UpDn", "UpDn", "UpDn", "UpDn", "UpDn", "UpDn", "Emp", "Emp", "Emp", "Emp", "Emp", "Emp", "Emp", "Emp"]
After sweep 1 energy=-160.39607113524528  maxlinkdim=4 maxerr=9.90E-07 time=0.281
After sweep 2 energy=-160.40131522275334  maxlinkdim=6 maxerr=8.85E-09 time=0.037
After sweep 3 energy=-160.40131539388884  maxlinkdim=9 maxerr=4.51E-10 time=0.058
After sweep 4 energy=-160.4013154020396  maxlinkdim=10 maxerr=9.85E-11 time=0.065
After sweep 5 energy=-160.40131541215254  maxlinkdim=12 maxerr=3.15E-13 time=0.076
After sweep 6 energy=-160.40131541215257  maxlinkdim=12 maxerr=3.15E-13 time=0.066
After sweep 7 energy=-160.40131541217758  maxlinkdim=14 maxerr=5.23E-15 time=0.089
After sweep 8 energy=-160.40131541217778  maxlinkdim=14 maxerr=5.23E-15 time=0.123
After sweep 9 energy=-160.40131541217755  maxlinkdim=14 maxerr=5.23E-15 time=0.096
After sweep 10 energy=-160.40131541217764  maxlinkdim=14 maxerr=5.23E-15 time=0.092
upd = [0.9749823571463203, 0.974701863185738

DomainError: DomainError with -5.675111337624178e-19:
log was called with a negative real argument but will only return a complex result if called with a complex argument. Try log(Complex(x)).